In [ ]:
import pandas as pd

file_path= "/content/final_query_suggestions_recovered_utf8sig.csv"

df= pd.read_csv(
    file_path,
    sep=";",
    dtype=str,
    encoding="utf-8-sig"
).fillna("")
print(df.head())

  English Term spanish term migrationsbezug_eng migrationsbezug_spa  \
0     A-Number     Número A                   1                   1   
1     A-Number     Número A                   1                   1   
2     A-Number     Número A                   1                   1   
3     A-Number     Número A                   1                   1   
4     A-Number     Número A                   1                   1   

  polysemie_eng polysemie_spa topic_number  \
0             0             0            3   
1             0             0            3   
2             0             0            3   
3             0             0            3   
4             0             0            3   

                                topic migrationsbezug_equal  \
0  Dokument, Visa, formelle Nachweise                     1   
1  Dokument, Visa, formelle Nachweise                     1   
2  Dokument, Visa, formelle Nachweise                     1   
3  Dokument, Visa, formelle Nachweise       

In [ ]:
usa_count = df[df['Land'] == 'USA'].shape[0]
print(f"Anzahl der Zeilen mit 'land=USA': {usa_count}")

Anzahl der Zeilen mit 'land=USA': 165614


In [ ]:
# Länder gruppieren
countries_dict = {'USA': 'usa', 'Mexiko': 'mexiko'}

# Alle übrigen Länder automatisch auf "andere" setzen
df['land_gruppe'] = df['Land'].map(countries_dict).fillna('andere')

# Themen identifizieren
topics = df['topic_number'].unique()

# Gruppennamen statt Original-Länder
country_groups = df['land_gruppe'].unique()

# Datensätze nach Ländergruppen und Themen aufteilen
for group in country_groups:
    for topic in topics:
        df_group_topic = df[
            (df['land_gruppe'] == group) &
            (df['topic_number'] == topic)
        ]

        df_name = f'df_{group}_thema_{topic}'
        globals()[df_name] = df_group_topic

        print(f"Datensatz {df_name} erstellt mit {df_group_topic.shape[0]} Zeilen.")

Datensatz df_andere_thema_3 erstellt mit 2380 Zeilen.
Datensatz df_andere_thema_5 erstellt mit 7974 Zeilen.
Datensatz df_andere_thema_4 erstellt mit 3258 Zeilen.
Datensatz df_andere_thema_1 erstellt mit 5067 Zeilen.
Datensatz df_andere_thema_2 erstellt mit 4422 Zeilen.
Datensatz df_mexiko_thema_3 erstellt mit 10880 Zeilen.
Datensatz df_mexiko_thema_5 erstellt mit 36432 Zeilen.
Datensatz df_mexiko_thema_4 erstellt mit 14906 Zeilen.
Datensatz df_mexiko_thema_1 erstellt mit 23193 Zeilen.
Datensatz df_mexiko_thema_2 erstellt mit 20214 Zeilen.
Datensatz df_usa_thema_3 erstellt mit 17000 Zeilen.
Datensatz df_usa_thema_5 erstellt mit 56876 Zeilen.
Datensatz df_usa_thema_4 erstellt mit 23466 Zeilen.
Datensatz df_usa_thema_1 erstellt mit 36638 Zeilen.
Datensatz df_usa_thema_2 erstellt mit 31634 Zeilen.


In [ ]:
MIGRATIONS_DEFINITION_DE = """
Im Rahmen dieser Arbeit bezeichnet Migrationsbezug den semantischen Bezug eines Suchterms oder einer Query Suggestion zu
grenzüberschreitender oder innerstaatlicher räumlicher Mobilität von Personen sowie zu damit verbundenen rechtlichen,
administrativen, sozialen oder politischen Prozessen.

Ein Begriff weist einen Migrationsbezug auf, wenn seine primäre und dominante Bedeutung unmittelbar auf Migration,
Aufenthaltsstatus, Grenzregime, Schutzformen, Staatsangehörigkeit oder migrationsbezogene Verwaltungsverfahren verweist
und außerhalb dieses Kontextes keine gebräuchliche Hauptbedeutung besitzt.

Auch wenn der Begriff ohne Bezug auf Migration semantisch nicht vollständig interpretierbar ist oder ein migrationsspezifisches
Phänomen oder Konzept bezeichnet, das ohne Migration keinen Sinn ergibt, einen rechtlichen Status oder Prozess im Migrationsrecht
beschreibt oder ein migrationspolitisches Steuerungsinstrument oder Institution im Migrationskontext meint, ist es als migrationsbezogen
zu betrachten.

Ein Begriff gilt nicht als migrationsbezogen, wenn er auch außerhalb von Migration in zentraler Bedeutung vorkommt oder nur in einem sehr
spezifischen Migrationsgesetz vorkommt, aber als Wort allgemein ist.

Polysemie:
- Wert 1: plausibel mehrere Bedeutungen im allgemeinen Sprachgebrauch, mindestens eine nicht migrationsbezogen -> ohne Kontext mehrdeutig.
- Wert 0: überwiegend eindeutig als Terminus im Migrationskontext (mehrgliedrige Fachphrasen, standardisierte Status-/Dokumentbezeichnungen,
  institutionelle Termini/Akronyme).
""".strip()

In [ ]:
# ============================================================
# 1) GEMINI API KEY PER PROMPT
# ============================================================

from getpass import getpass
import os
from google import genai

os.environ["GOOGLE_API_KEY"] = getpass("Gemini API Key eingeben: ").strip()

MODEL = "gemini-3-flash-preview"

client = genai.Client()  # liest GOOGLE_API_KEY automatisch
print("✅ Gemini Client initialisiert")

Gemini API Key eingeben: ··········
✅ Gemini Client initialisiert


In [ ]:
import os
import re
import json
import time
import hashlib
import unicodedata
from typing import Any, Dict, List

import pandas as pd

from google import genai
from google.genai import types


# ============================================================
# 0) VORAUSSETZUNGEN
# ============================================================
# Erwartet:
# - df_usa_thema_3 existiert bereits als pandas DataFrame
# - client existiert bereits, z. B.:
#     client = genai.Client()
# - MODEL existiert bereits, z. B.:
#     MODEL = "gemini-3-flash-preview"
# - MIGRATIONS_DEFINITION_DE existiert bereits als String

# Beispiel, falls noch nicht gesetzt:
# MODEL = "gemini-3-flash-preview"
# client = genai.Client()

OUTPUT_DIR = "qs_label_outputs_usa_thema_4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 20
MAX_RETRIES = 3
RETRY_SLEEP = 2.0


# ============================================================
# 1) HILFSFUNKTIONEN
# ============================================================

def normalize_text(text: Any) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def sha1_hex(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def make_qs_identifier(qs_text: str, lang: str) -> str:
    qs_norm = normalize_text(qs_text)
    return f"{lang}_{sha1_hex(qs_norm)}"


def make_term_qs_identifier(term_text: str, qs_text: str, lang: str) -> str:
    term_norm = normalize_text(term_text)
    qs_norm = normalize_text(qs_text)
    combo = f"{lang}||{term_norm}||{qs_norm}"
    return f"{lang}_tq_{sha1_hex(combo)}"


def safe_to01(x: Any) -> str:
    return "1" if str(x).strip() == "1" else "0"


def chunk_dataframe(df: pd.DataFrame, batch_size: int) -> List[pd.DataFrame]:
    return [df.iloc[i:i + batch_size].copy() for i in range(0, len(df), batch_size)]


def extract_json_text(response) -> Any:
    text = getattr(response, "text", "") or ""
    text = text.strip()

    if not text:
        raise ValueError("Leere Modellantwort.")

    decoder = json.JSONDecoder()

    try:
        obj, end = decoder.raw_decode(text)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            return [obj]
    except Exception:
        pass

    start = text.find("[")
    end = text.rfind("]")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except Exception:
            pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            obj = json.loads(text[start:end + 1])
            if isinstance(obj, dict):
                return [obj]
            if isinstance(obj, list):
                return obj
        except Exception:
            pass

    raise ValueError(f"Keine gültige JSON-Antwort parsebar. Rohtext:\n{text[:2000]}")


# ============================================================
# 2) SPALTEN STANDARDISIEREN
# ============================================================

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]

    rename_map = {
        "English Term": "english_term",
        "english term": "english_term",

        "Spanish Term": "spanish_term",
        "spanish term": "spanish_term",

        "migrationsbezug_eng": "migrationsbezug_eng",
        "migrationsbezug_spa": "migrationsbezug_spa",

        "polysemie_eng": "polysemie_eng",
        "Polysemie_eng": "polysemie_eng",

        "polysemie_spa": "polysemie_spa",
        "Polysemie_spa": "polysemie_spa",

        "topic_number": "topic_number",
        "topic": "topic",

        "migrationsbezug_equal": "migrationsbezug_equal",
        "migrationsbezug_any": "migrationsbezug_any",
        "Migrationsbezug_any": "migrationsbezug_any",
        "migrationsbezug_both": "migrationsbezug_both",

        "QS Position": "qs_position",
        "QS position": "qs_position",

        "QS eng": "qs_eng",
        "QS spa": "qs_spa",

        "Bundesstaat": "bundesstaat",
        "bundesstaat": "bundesstaat",

        "Stadt": "stadt",
        "stadt": "stadt",

        "Land": "land",
        "Uhrzeit": "uhrzeit",
        "Uhrzeit (scraping zeitpunkt)": "uhrzeit",
    }

    out = out.rename(columns=rename_map)

    required_cols = [
        "english_term",
        "spanish_term",
        "migrationsbezug_eng",
        "migrationsbezug_spa",
        "polysemie_eng",
        "polysemie_spa",
        "qs_position",
        "qs_eng",
        "qs_spa",
    ]

    missing = [c for c in required_cols if c not in out.columns]
    if missing:
        raise ValueError(
            f"Fehlende Spalten nach rename: {missing}\n"
            f"Vorhandene Spalten: {out.columns.tolist()}"
        )

    return out


# ============================================================
# 3) UNIQUE TABELLEN PRO SPRACHE
# ============================================================

def prepare_language_table(df_original: pd.DataFrame, lang: str) -> pd.DataFrame:
    if lang not in {"en", "es"}:
        raise ValueError("lang muss 'en' oder 'es' sein.")

    work = df_original.copy()

    if lang == "en":
        work = work[
            ["english_term", "migrationsbezug_eng", "polysemie_eng", "qs_position", "qs_eng"]
        ].copy()
        work = work.rename(columns={
            "english_term": "term",
            "migrationsbezug_eng": "migrationsbezug_term",
            "polysemie_eng": "polysemie_term",
            "qs_eng": "qs",
        })
    else:
        work = work[
            ["spanish_term", "migrationsbezug_spa", "polysemie_spa", "qs_position", "qs_spa"]
        ].copy()
        work = work.rename(columns={
            "spanish_term": "term",
            "migrationsbezug_spa": "migrationsbezug_term",
            "polysemie_spa": "polysemie_term",
            "qs_spa": "qs",
        })

    work["term_norm"] = work["term"].map(normalize_text)
    work["qs_norm"] = work["qs"].map(normalize_text)

    work = work[(work["term_norm"] != "") & (work["qs_norm"] != "")].copy()

    work["language"] = lang
    work["qs_identifier"] = [make_qs_identifier(q, lang) for q in work["qs"]]
    work["term_qs_identifier"] = [
        make_term_qs_identifier(t, q, lang)
        for t, q in zip(work["term"], work["qs"])
    ]

    # Nur einmal pro Term x QS
    work = work.drop_duplicates(subset=["term_norm", "qs_norm"]).copy()

    work["qs_migrationsbezug"] = ""
    work["status"] = ""
    work["error_message"] = ""

    return work[
        [
            "language",
            "term",
            "migrationsbezug_term",
            "polysemie_term",
            "qs_position",
            "qs",
            "qs_identifier",
            "term_qs_identifier",
            "term_norm",
            "qs_norm",
            "qs_migrationsbezug",
            "status",
            "error_message",
        ]
    ].reset_index(drop=True)


# ============================================================
# 4) PROMPT + SCHEMA
# ============================================================

def build_records_for_batch(df_batch: pd.DataFrame) -> List[Dict[str, Any]]:
    return [
        {
            "id": row["term_qs_identifier"],
            "term": row["term"],
            "migrationsbezug_term": safe_to01(row["migrationsbezug_term"]),
            "polysemie_term": safe_to01(row["polysemie_term"]),
            "qs": row["qs"],
        }
        for _, row in df_batch.iterrows()
    ]


def build_prompt(records: List[Dict[str, Any]], lang: str) -> str:
    language_label = "English" if lang == "en" else "Spanish"

    return f"""
Du bist ein Labeling-Assistent für Query Suggestions.
Gib ausschließlich gültiges JSON zurück. Kein Markdown. Keine Erklärungen.

Definition von Migrationsbezug:
{MIGRATIONS_DEFINITION_DE}

Aufgabe:
Bewerte für jeden Datensatz das finale Label "qs_migrationsbezug" als 0 oder 1.

Bedeutung des Labels:
- qs_migrationsbezug = 1, wenn die GESAMTKOMBINATION aus
  (Term + Query Suggestion) gemäß der obigen Definition migrationsbezogen ist.
- qs_migrationsbezug = 0, wenn die Gesamtkombination gemäß der obigen Definition
  nicht migrationsbezogen ist.

Vorhandene Signale:
- migrationsbezug_term = vorhandenes Label für das Basiswort allein
- polysemie_term = vorhandenes Polysemie-Label für das Basiswort allein

Wichtige Regeln:
- migrationsbezug_term soll berücksichtigt werden, aber nicht blind übernommen werden.
- Das finale Label bezieht sich ausschließlich auf die Kombination aus Term und Query Suggestion.
- Es ist möglich, dass migrationsbezug_term = 0, aber qs_migrationsbezug = 1,
  wenn die Query Suggestion den Migrationsbezug erst herstellt.
- Es ist möglich, dass migrationsbezug_term = 1, aber qs_migrationsbezug = 0,
  wenn die Query Suggestion den Migrationsbezug verliert oder in eine andere Bedeutung verschiebt.
- Lexikalische oder Übersetzungsfragen zählen NICHT als Verstärker des Migrationsbezugs.
- Insbesondere Formulierungen wie "in english", "meaning", "definition", "what does ... mean",
  "en inglés", "significado", "definición" oder ähnliche lexikalische Nachfragen sollen
  NICHT als Verstärkung des Migrationsbezugs bewertet werden.
- Wenn die Query Suggestion nur nach Übersetzung, Bedeutung, Definition oder Worterklärung fragt,
  dann ist das eine lexikalische Frage und kein zusätzlicher migrationsbezogener Verstärker.
- Beurteile alles nur in der Sprache: {language_label}
- Antworte nur mit einem JSON-Array.
- Gib genau ein Objekt pro Input-Datensatz zurück.
- Jedes Objekt darf nur diese Keys enthalten:
  - id
  - qs_migrationsbezug

Beispiele:
Term: "asilo"
migrationsbezug_term: 1
qs: "restaurant"
=> qs_migrationsbezug = 0

Term: "asilo"
migrationsbezug_term: 1
qs: "estados unidos"
=> qs_migrationsbezug = 1

Term: "displacement"
migrationsbezug_term: 0
polysemie_term: 1
qs: "refugees"
=> qs_migrationsbezug = 1

Term: "asylum"
migrationsbezug_term: 1
qs: "meaning"
=> qs_migrationsbezug = 0

Term: "asilo"
migrationsbezug_term: 1
qs: "significado"
=> qs_migrationsbezug = 0

Datensätze:
{json.dumps(records, ensure_ascii=False, separators=(",", ":"))}
""".strip()


def get_response_schema():
    return {
        "type": "array",
        "items": {
            "type": "object",
            "properties": {
                "id": {"type": "string"},
                "qs_migrationsbezug": {"type": "string", "enum": ["0", "1"]},
            },
            "required": ["id", "qs_migrationsbezug"],
        },
    }


# ============================================================
# 5) LLM CALLS
# ============================================================

def call_model_for_batch(df_batch: pd.DataFrame, lang: str) -> List[Dict[str, Any]]:
    records = build_records_for_batch(df_batch)
    prompt = build_prompt(records, lang)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=get_response_schema(),
        ),
    )

    parsed = getattr(response, "parsed", None)
    if parsed is not None:
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            return [parsed]

    parsed = extract_json_text(response)
    if isinstance(parsed, dict):
        return [parsed]
    if isinstance(parsed, list):
        return parsed

    raise ValueError(f"Unerwartetes Response-Format: {type(parsed)}")


def label_language_table(df_unique: pd.DataFrame, lang: str) -> pd.DataFrame:
    out = df_unique.copy()
    batches = chunk_dataframe(out, BATCH_SIZE)

    print(f"[{lang}] Starte {len(batches)} Batches")

    batch_results = []

    for i, df_batch in enumerate(batches, start=1):
        last_err = None
        batch_result = df_batch.copy()

        for attempt in range(MAX_RETRIES):
            try:
                result = call_model_for_batch(df_batch, lang)

                result_map = {}
                for item in result:
                    rid = str(item.get("id", "")).strip()
                    lbl = safe_to01(item.get("qs_migrationsbezug"))
                    if rid:
                        result_map[rid] = lbl

                for idx, row in batch_result.iterrows():
                    rid = row["term_qs_identifier"]
                    if rid in result_map:
                        batch_result.at[idx, "qs_migrationsbezug"] = result_map[rid]
                        batch_result.at[idx, "status"] = "ok"
                        batch_result.at[idx, "error_message"] = ""
                    else:
                        batch_result.at[idx, "status"] = "error"
                        batch_result.at[idx, "error_message"] = "Missing result for id"

                last_err = None
                break

            except Exception as e:
                last_err = e
                print(f"[{lang}] Batch {i}/{len(batches)} Fehler Versuch {attempt+1}/{MAX_RETRIES}: {e}")
                time.sleep(RETRY_SLEEP * (attempt + 1))

        if last_err is not None:
            for idx in batch_result.index:
                batch_result.at[idx, "status"] = "error"
                batch_result.at[idx, "error_message"] = str(last_err)

        batch_results.append(batch_result)

        if i % 5 == 0 or i == len(batches):
            print(f"[{lang}] Fertig mit {i}/{len(batches)} Batches")

    return pd.concat(batch_results, ignore_index=True)


# ============================================================
# 6) ZURÜCKMERGEN
# ============================================================

def add_identifiers_to_original(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["qs_identifier_en"] = [make_qs_identifier(x, "en") for x in out["qs_eng"]]
    out["term_qs_identifier_en"] = [
        make_term_qs_identifier(term, qs, "en")
        for term, qs in zip(out["english_term"], out["qs_eng"])
    ]

    out["qs_identifier_es"] = [make_qs_identifier(x, "es") for x in out["qs_spa"]]
    out["term_qs_identifier_es"] = [
        make_term_qs_identifier(term, qs, "es")
        for term, qs in zip(out["spanish_term"], out["qs_spa"])
    ]

    return out


def merge_labels_back(
    df_original: pd.DataFrame,
    df_en_labeled: pd.DataFrame,
    df_es_labeled: pd.DataFrame,
) -> pd.DataFrame:
    df = add_identifiers_to_original(df_original)

    en_map = df_en_labeled[
        ["term_qs_identifier", "qs_identifier", "qs_migrationsbezug", "status", "error_message"]
    ].rename(columns={
        "term_qs_identifier": "term_qs_identifier_en",
        "qs_identifier": "qs_identifier_en",
        "qs_migrationsbezug": "qs_migrationsbezug_eng_final",
        "status": "status_eng_final",
        "error_message": "error_message_eng_final",
    })

    es_map = df_es_labeled[
        ["term_qs_identifier", "qs_identifier", "qs_migrationsbezug", "status", "error_message"]
    ].rename(columns={
        "term_qs_identifier": "term_qs_identifier_es",
        "qs_identifier": "qs_identifier_es",
        "qs_migrationsbezug": "qs_migrationsbezug_spa_final",
        "status": "status_spa_final",
        "error_message": "error_message_spa_final",
    })

    df = df.merge(en_map, on=["term_qs_identifier_en", "qs_identifier_en"], how="left")
    df = df.merge(es_map, on=["term_qs_identifier_es", "qs_identifier_es"], how="left")

    return df


# ============================================================
# 7) MAIN WORKFLOW FÜR df_usa_thema_4
# ============================================================

df_base = standardize_columns(df_usa_thema_4.copy())

print("Zeilen im Input:", len(df_base))

# UNIQUE DATEIEN
df_en_unique = prepare_language_table(df_base, "en")
df_es_unique = prepare_language_table(df_base, "es")

print("Eindeutige EN-Kombinationen:", len(df_en_unique))
print("Eindeutige ES-Kombinationen:", len(df_es_unique))

en_unique_path = os.path.join(OUTPUT_DIR, "usa_thema_4_english_term_qs_unique.csv")
es_unique_path = os.path.join(OUTPUT_DIR, "usa_thema_4_spanish_term_qs_unique.csv")

df_en_unique.to_csv(en_unique_path, index=False, encoding="utf-8-sig")
df_es_unique.to_csv(es_unique_path, index=False, encoding="utf-8-sig")

print("Gespeichert:", en_unique_path)
print("Gespeichert:", es_unique_path)

# LABELING
print("\nStarte EN-Labeling ...")
df_en_labeled = label_language_table(df_en_unique, "en")

print("\nStarte ES-Labeling ...")
df_es_labeled = label_language_table(df_es_unique, "es")

en_labeled_path = os.path.join(OUTPUT_DIR, "usa_thema_4_english_term_qs_labeled.csv")
es_labeled_path = os.path.join(OUTPUT_DIR, "usa_thema_4_spanish_term_qs_labeled.csv")

df_en_labeled.to_csv(en_labeled_path, index=False, encoding="utf-8-sig")
df_es_labeled.to_csv(es_labeled_path, index=False, encoding="utf-8-sig")

print("Gespeichert:", en_labeled_path)
print("Gespeichert:", es_labeled_path)

# MERGE BACK
df_merged = merge_labels_back(df_base, df_en_labeled, df_es_labeled)

merged_path = os.path.join(OUTPUT_DIR, "usa_thema_4_with_qs_labels.csv")
df_merged.to_csv(merged_path, index=False, encoding="utf-8-sig")

print("Gespeichert:", merged_path)

print("\nEN Status:")
print(df_en_labeled["status"].value_counts(dropna=False))

print("\nES Status:")
print(df_es_labeled["status"].value_counts(dropna=False))

Zeilen im Input: 23466
Eindeutige EN-Kombinationen: 1952
Eindeutige ES-Kombinationen: 901
Gespeichert: qs_label_outputs_usa_thema_4/usa_thema_4_english_term_qs_unique.csv
Gespeichert: qs_label_outputs_usa_thema_4/usa_thema_4_spanish_term_qs_unique.csv

Starte EN-Labeling ...
[en] Starte 98 Batches
[en] Fertig mit 5/98 Batches
[en] Fertig mit 10/98 Batches
[en] Fertig mit 15/98 Batches
[en] Fertig mit 20/98 Batches
[en] Fertig mit 25/98 Batches
[en] Fertig mit 30/98 Batches
[en] Fertig mit 35/98 Batches
[en] Fertig mit 40/98 Batches
[en] Fertig mit 45/98 Batches
[en] Fertig mit 50/98 Batches
[en] Fertig mit 55/98 Batches
[en] Fertig mit 60/98 Batches
[en] Fertig mit 65/98 Batches
[en] Fertig mit 70/98 Batches
[en] Fertig mit 75/98 Batches
[en] Fertig mit 80/98 Batches
[en] Fertig mit 85/98 Batches
[en] Fertig mit 90/98 Batches
[en] Fertig mit 95/98 Batches
[en] Fertig mit 98/98 Batches

Starte ES-Labeling ...
[es] Starte 46 Batches
[es] Fertig mit 5/46 Batches
[es] Fertig mit 10/46 Batc

In [ ]:
df_final = merge_labels_back(df_base, df_en_labeled, df_es_labeled)

In [ ]:
output_path = "labeled_usa_thema_4_final.csv"

df_final.to_csv(
    output_path,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"Gespeichert unter: {output_path}")

Gespeichert unter: labeled_usa_thema_4_final.csv


In [ ]:
# ============================================================
# MEXIKO THEMA 3
# ============================================================

OUTPUT_DIR = "qs_label_outputs_mexiko_thema_4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 20
MAX_RETRIES = 3
RETRY_SLEEP = 2.0

# 1) Basis-DF standardisieren
df_base_mexiko = standardize_columns(df_mexiko_thema_4.copy())

print("Zeilen im Input:", len(df_base_mexiko))

# 2) Unique-Dateien für beide Sprachen bauen
df_en_unique_mexiko = prepare_language_table(df_base_mexiko, "en")
df_es_unique_mexiko = prepare_language_table(df_base_mexiko, "es")

print("Eindeutige EN-Kombinationen:", len(df_en_unique_mexiko))
print("Eindeutige ES-Kombinationen:", len(df_es_unique_mexiko))

# 3) Unique-Dateien speichern
en_unique_path = os.path.join(OUTPUT_DIR, "mexiko_thema_4_english_term_qs_unique.csv")
es_unique_path = os.path.join(OUTPUT_DIR, "mexiko_thema_4_spanish_term_qs_unique.csv")

df_en_unique_mexiko.to_csv(en_unique_path, index=False, sep=";", encoding="utf-8-sig")
df_es_unique_mexiko.to_csv(es_unique_path, index=False, sep=";", encoding="utf-8-sig")

print("Gespeichert:", en_unique_path)
print("Gespeichert:", es_unique_path)

# 4) Labeling
print("\nStarte EN-Labeling ...")
df_en_labeled_mexiko = label_language_table(df_en_unique_mexiko, "en")

print("\nStarte ES-Labeling ...")
df_es_labeled_mexiko = label_language_table(df_es_unique_mexiko, "es")

# 5) Gelabelte Unique-Dateien speichern
en_labeled_path = os.path.join(OUTPUT_DIR, "mexiko_thema_4_english_term_qs_labeled.csv")
es_labeled_path = os.path.join(OUTPUT_DIR, "mexiko_thema_4_spanish_term_qs_labeled.csv")

df_en_labeled_mexiko.to_csv(en_labeled_path, index=False, sep=";", encoding="utf-8-sig")
df_es_labeled_mexiko.to_csv(es_labeled_path, index=False, sep=";", encoding="utf-8-sig")

print("Gespeichert:", en_labeled_path)
print("Gespeichert:", es_labeled_path)

# 6) Zurück in den Original-Datensatz mergen
df_mexiko_thema_4_final = merge_labels_back(
    df_base_mexiko,
    df_en_labeled_mexiko,
    df_es_labeled_mexiko,
)

# 7) Final speichern
final_path = "labeled_mexiko_thema_4_final.csv"

df_mexiko_thema_4_final.to_csv(
    final_path,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Gespeichert:", final_path)

# 8) Kurzer Statuscheck
print("\nEN Status:")
print(df_en_labeled_mexiko["status"].value_counts(dropna=False))

print("\nES Status:")
print(df_es_labeled_mexiko["status"].value_counts(dropna=False))

Zeilen im Input: 14906
Eindeutige EN-Kombinationen: 641
Eindeutige ES-Kombinationen: 1196
Gespeichert: qs_label_outputs_mexiko_thema_4/mexiko_thema_4_english_term_qs_unique.csv
Gespeichert: qs_label_outputs_mexiko_thema_4/mexiko_thema_4_spanish_term_qs_unique.csv

Starte EN-Labeling ...
[en] Starte 33 Batches
[en] Fertig mit 5/33 Batches
[en] Fertig mit 10/33 Batches
[en] Fertig mit 15/33 Batches
[en] Fertig mit 20/33 Batches
[en] Fertig mit 25/33 Batches
[en] Fertig mit 30/33 Batches
[en] Fertig mit 33/33 Batches

Starte ES-Labeling ...
[es] Starte 60 Batches
[es] Fertig mit 5/60 Batches
[es] Fertig mit 10/60 Batches
[es] Fertig mit 15/60 Batches
[es] Fertig mit 20/60 Batches
[es] Fertig mit 25/60 Batches
[es] Fertig mit 30/60 Batches
[es] Fertig mit 35/60 Batches
[es] Fertig mit 40/60 Batches
[es] Fertig mit 45/60 Batches
[es] Fertig mit 50/60 Batches
[es] Fertig mit 55/60 Batches
[es] Fertig mit 60/60 Batches
Gespeichert: qs_label_outputs_mexiko_thema_4/mexiko_thema_4_english_term_q